# 🚌 ETA Prediction – Bus Transjakarta
**Target**: `eta_minutes` = selisih waktu antara `tapOutTime` dan `tapInTime`  
**Input features**: `corridorID`, `tapInStops`, `tapOutStops`, `hour`, `day_of_week`, `direction`, `stopStartSeq`, `stopEndSeq`  
**Models**: Random Forest & XGBoost  

In [ ]:
# ── Install deps (run sekali aja) ──────────────────────────────────
# !pip install xgboost shap -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110
SEED = 42
print('Libraries loaded ✅')

---
## 1. Load & Merge Data

In [ ]:
besar = pd.read_csv('bus_besar_all.csv')
kecil = pd.read_csv('bus_kecil_all.csv')

besar['bus_type'] = 'besar'
kecil['bus_type'] = 'kecil'

df = pd.concat([besar, kecil], ignore_index=True)
print(f'Total rows  : {len(df):,}')
print(f'Bus Besar   : {len(besar):,}')
print(f'Bus Kecil   : {len(kecil):,}')
df.head(3)

---
## 2. Feature Engineering

In [ ]:
# Parse timestamps
df['tapInTime']  = pd.to_datetime(df['tapInTime'],  errors='coerce')
df['tapOutTime'] = pd.to_datetime(df['tapOutTime'], errors='coerce')

# Target: ETA dalam menit
df['eta_minutes'] = (df['tapOutTime'] - df['tapInTime']).dt.total_seconds() / 60

# Time features dari tapInTime
df['hour']        = df['tapInTime'].dt.hour
df['day_of_week'] = df['tapInTime'].dt.dayofweek   # 0=Senin, 6=Minggu
df['is_weekend']  = (df['day_of_week'] >= 5).astype(int)

# Stop sequence distance
df['stop_dist'] = (df['stopEndSeq'] - df['stopStartSeq']).abs()

print('ETA stats (menit):')
print(df['eta_minutes'].describe().round(2))

In [ ]:
# Drop baris yang tidak punya tapOutTime / eta negatif / outlier ekstrem
before = len(df)
df = df.dropna(subset=['tapOutTime', 'tapInStops', 'tapOutStops', 'stopEndSeq'])
df = df[df['eta_minutes'] > 0]
df = df[df['eta_minutes'] <= 180]   # cap 3 jam – outlier transport aneh
after = len(df)
print(f'Rows setelah cleaning: {after:,}  (dihapus {before-after:,})')

---
## 3. EDA

In [ ]:
# 3.1 Distribusi ETA
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, btype, color in zip(axes, ['besar', 'kecil'], ['steelblue', 'coral']):
    subset = df[df['bus_type'] == btype]['eta_minutes']
    ax.hist(subset, bins=60, color=color, edgecolor='white', alpha=0.85)
    ax.set_title(f'Distribusi ETA – Bus {btype.capitalize()}', fontweight='bold')
    ax.set_xlabel('ETA (menit)')
    ax.set_ylabel('Frekuensi')
    ax.axvline(subset.median(), color='black', ls='--', label=f'Median {subset.median():.1f} mnt')
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# 3.2 ETA per jam
fig, ax = plt.subplots(figsize=(13, 4))
for btype, color in [('besar', 'steelblue'), ('kecil', 'coral')]:
    sub = df[df['bus_type'] == btype].groupby('hour')['eta_minutes'].median()
    ax.plot(sub.index, sub.values, marker='o', label=f'Bus {btype}', color=color)

ax.set_title('Median ETA per Jam (tapInTime)', fontweight='bold')
ax.set_xlabel('Jam')
ax.set_ylabel('ETA Median (menit)')
ax.set_xticks(range(0, 24))
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 3.3 ETA per hari dalam seminggu
day_labels = ['Sen','Sel','Rab','Kam','Jum','Sab','Min']
fig, ax = plt.subplots(figsize=(10, 4))
for btype, color in [('besar', 'steelblue'), ('kecil', 'coral')]:
    sub = df[df['bus_type'] == btype].groupby('day_of_week')['eta_minutes'].median()
    ax.bar(sub.index + (0 if btype == 'besar' else 0.35),
           sub.values, width=0.35, label=f'Bus {btype}', color=color, alpha=0.85)

ax.set_xticks([i + 0.175 for i in range(7)])
ax.set_xticklabels(day_labels)
ax.set_title('Median ETA per Hari', fontweight='bold')
ax.set_ylabel('ETA Median (menit)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 3.4 ETA vs stop_dist
fig, ax = plt.subplots(figsize=(8, 4))
for btype, color in [('besar', 'steelblue'), ('kecil', 'coral')]:
    sub = df[df['bus_type'] == btype]
    ax.scatter(sub['stop_dist'], sub['eta_minutes'],
               alpha=0.15, s=5, color=color, label=f'Bus {btype}')

ax.set_title('ETA vs Selisih Nomor Stop', fontweight='bold')
ax.set_xlabel('Selisih Stop (stopEndSeq - stopStartSeq)')
ax.set_ylabel('ETA (menit)')
ax.legend(markerscale=5)
plt.tight_layout()
plt.show()

In [ ]:
# 3.5 Top-10 corridor by volume
top10 = df.groupby('corridorID')['eta_minutes'].agg(['count','median']).sort_values('count', ascending=False).head(10)
print(top10.rename(columns={'count':'jumlah_transaksi', 'median':'eta_median_mnt'}).round(1))

In [ ]:
# 3.6 Heatmap korelasi numerik
num_cols = ['eta_minutes','hour','day_of_week','direction','stopStartSeq','stopEndSeq','stop_dist','is_weekend']
corr = df[num_cols].corr()
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax, linewidths=0.5)
ax.set_title('Korelasi Fitur Numerik', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. Preprocessing untuk Modelling

In [ ]:
FEATURES = ['corridorID', 'tapInStops', 'tapOutStops',
            'hour', 'day_of_week', 'direction',
            'stopStartSeq', 'stopEndSeq',
            'bus_type']   # tambahan kolom jenis bus
TARGET = 'eta_minutes'

model_df = df[FEATURES + [TARGET]].dropna().copy()

# Label encode kolom kategorik
cat_cols = ['corridorID', 'tapInStops', 'tapOutStops', 'bus_type']
encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    model_df[col] = le.fit_transform(model_df[col].astype(str))
    encoders[col] = le

print(f'Dataset model: {model_df.shape}')
model_df.head(3)

In [ ]:
X = model_df[FEATURES]
y = model_df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)
print(f'Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}')

---
## 5. Model Training
### 5.1 Random Forest

In [ ]:
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=SEED
)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
print('Random Forest trained ✅')

### 5.2 XGBoost

In [ ]:
xgb_model = xgb.XGBRegressor(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    tree_method='hist',
    random_state=SEED,
    verbosity=0
)
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)
y_pred_xgb = xgb_model.predict(X_test)
print('XGBoost trained ✅')

---
## 6. Evaluasi

In [ ]:
def evaluate(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    r2   = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true.clip(lower=1))) * 100
    print(f'[{name}]')
    print(f'  MAE  : {mae:.2f} menit')
    print(f'  RMSE : {rmse:.2f} menit')
    print(f'  MAPE : {mape:.1f}%')
    print(f'  R²   : {r2:.4f}')
    print()
    return {'model': name, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape, 'R2': r2}

results = []
results.append(evaluate('Random Forest', y_test, y_pred_rf))
results.append(evaluate('XGBoost',       y_test, y_pred_xgb))

In [ ]:
# Tabel perbandingan
results_df = pd.DataFrame(results).set_index('model')
results_df.round(3)

In [ ]:
# Bar chart perbandingan MAE & RMSE
metrics = ['MAE', 'RMSE']
x = np.arange(len(metrics))
width = 0.3

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x - width/2, results_df.loc['Random Forest', metrics], width, label='Random Forest', color='steelblue')
ax.bar(x + width/2, results_df.loc['XGBoost',       metrics], width, label='XGBoost',       color='coral')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylabel('Menit')
ax.set_title('Perbandingan Error – RF vs XGBoost', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Scatter: Actual vs Predicted
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, name, preds, color in [
    (axes[0], 'Random Forest', y_pred_rf,  'steelblue'),
    (axes[1], 'XGBoost',       y_pred_xgb, 'coral')
]:
    ax.scatter(y_test, preds, alpha=0.2, s=5, color=color)
    lims = [0, max(y_test.max(), preds.max())]
    ax.plot(lims, lims, 'k--', lw=1)
    ax.set_title(f'{name} – Actual vs Predicted', fontweight='bold')
    ax.set_xlabel('Actual ETA (menit)')
    ax.set_ylabel('Predicted ETA (menit)')

plt.tight_layout()
plt.show()

---
## 7. Feature Importance

In [ ]:
def plot_importance(model, feature_names, title, color):
    imp = pd.Series(model.feature_importances_, index=feature_names).sort_values()
    fig, ax = plt.subplots(figsize=(8, 5))
    imp.plot(kind='barh', color=color, ax=ax)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Importance')
    plt.tight_layout()
    plt.show()

plot_importance(rf,        FEATURES, 'Feature Importance – Random Forest', 'steelblue')
plot_importance(xgb_model, FEATURES, 'Feature Importance – XGBoost',       'coral')

---
## 8. Residual Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, name, preds, color in [
    (axes[0], 'Random Forest', y_pred_rf,  'steelblue'),
    (axes[1], 'XGBoost',       y_pred_xgb, 'coral')
]:
    residuals = y_test.values - preds
    ax.scatter(preds, residuals, alpha=0.2, s=5, color=color)
    ax.axhline(0, color='black', lw=1, ls='--')
    ax.set_title(f'Residuals – {name}', fontweight='bold')
    ax.set_xlabel('Predicted ETA (menit)')
    ax.set_ylabel('Residual (Actual - Predicted)')

plt.tight_layout()
plt.show()

---
## 9. Prediksi pada Data Baru
Contoh cara pakai model untuk prediksi single trip.

In [ ]:
def predict_eta(corridorID, tapInStops, tapOutStops,
                hour, day_of_week, direction,
                stopStartSeq, stopEndSeq,
                bus_type='besar',
                model=xgb_model):
    """
    Prediksi ETA (menit) untuk satu perjalanan.
    Semua string input harus match dengan data training.
    """
    row = {
        'corridorID'  : corridorID,
        'tapInStops'  : tapInStops,
        'tapOutStops' : tapOutStops,
        'hour'        : hour,
        'day_of_week' : day_of_week,
        'direction'   : direction,
        'stopStartSeq': stopStartSeq,
        'stopEndSeq'  : stopEndSeq,
        'bus_type'    : bus_type,
    }
    sample = pd.DataFrame([row])
    for col in cat_cols:
        le = encoders[col]
        val = sample[col].astype(str).values[0]
        if val in le.classes_:
            sample[col] = le.transform([val])
        else:
            sample[col] = -1  # unknown category
    
    eta = model.predict(sample[FEATURES])[0]
    return round(eta, 1)

# --- Contoh prediksi ---
# Ganti nilai di bawah dengan koridor & stop yang valid dari dataset kamu
sample_corridorID   = df['corridorID'].iloc[0]
sample_tapInStops   = df['tapInStops'].iloc[0]
sample_tapOutStops  = df['tapOutStops'].iloc[0]
sample_stopStartSeq = int(df['stopStartSeq'].iloc[0])
sample_stopEndSeq   = int(df['stopEndSeq'].iloc[0])

eta_est = predict_eta(
    corridorID   = sample_corridorID,
    tapInStops   = sample_tapInStops,
    tapOutStops  = sample_tapOutStops,
    hour         = 8,       # jam 08:00
    day_of_week  = 0,       # Senin
    direction    = 1,
    stopStartSeq = sample_stopStartSeq,
    stopEndSeq   = sample_stopEndSeq,
    bus_type     = 'besar',
    model        = xgb_model
)

print(f'Prediksi ETA  : {eta_est} menit')
print(f'Koridor       : {sample_corridorID}')
print(f'Tap In Stop   : {sample_tapInStops}')
print(f'Tap Out Stop  : {sample_tapOutStops}')

---
## 10. Summary & Next Steps

| Langkah | Status |
|---------|--------|
| Merge bus besar & kecil + kolom `bus_type` | ✅ |
| Feature engineering (hour, day_of_week, stop_dist) | ✅ |
| EDA – distribusi, tren jam, korelasi | ✅ |
| Baseline RF & XGBoost | ✅ |
| Feature importance & residual analysis | ✅ |

### 💡 Ide Improvement
- **Hyperparameter tuning** – `GridSearchCV` atau `Optuna` untuk RF & XGBoost
- **Geospatial features** – Haversine distance antara tapInStopsLat/Lon dan tapOutStopsLat/Lon
- **Traffic proxy** – encode `hour × day_of_week` sebagai rush-hour category
- **Target encoding** – encode `corridorID` pakai rata-rata ETA per koridor (lebih informatif dari label encoding)
- **Stacking / blending** – gabungkan prediksi RF + XGBoost
- **LightGBM** – biasanya lebih cepat dan akurat untuk data kategorikal besar
- **Cross-validation** – `TimeSeriesSplit` untuk split yang respect urutan waktu
